# Group 2 — Which explanation do we trust, and where in the flow?

**AI Capstone project — finding important turbulence-model parameters with explainable AI**

---

## The engineering problem

Computational Fluid Dynamics (CFD) predicts turbulent flows using a *turbulence
model*. The model used here, **SST $k$–$\omega$**, contains nine empirical
constants. Nobody knows their exact values; each is only known to lie inside an
interval. Different choices inside those intervals give different predicted
flows, so an engineer who wants a trustworthy prediction needs to know **which of
the nine constants actually matter**.

The reference answer comes from a *Sobol sensitivity analysis*: 1408 CFD
simulations per flow case, decomposing the output variance among the nine
parameters. That answer is given to you as a **benchmark** — you never compute it
yourself.

## Your question

A machine-learning model can be asked "which input mattered?" in several
different ways, and the ways do not always agree. Worse, the answer depends on
*which output you ask about*: the parameter that controls the pressure
fluctuation at the wall need not be the one that controls turbulent energy in the
middle of the channel.

> Which explainability method should an engineer trust, and how does parameter
> importance change across the flow?

This is what Group 2 investigates. You will:

1. train a surrogate model and explain it with **several different methods**;
2. check those explanations against the Sobol benchmark;
3. extend from a single number to **every one of the 80 CFD outputs**, and map
   where in the flow each parameter matters.

---

---

## How to use this notebook

This is the **skeleton of your project**, not a worked example. The explanatory
text, the data loading and the figures that show you the provided data are
given. Everything else is yours.

Look for `# TODO` blocks. Each states a task and names the variables that later
cells expect. They deliberately do **not** give you the exact function call:
[`RESOURCES.md`](../RESOURCES.md) lists every method you need with a link to its
documentation and a runnable example, and choosing your own settings — then
being able to say why — is part of what you are assessed on.

Sections are labelled **MUST**, **SHOULD** or **COULD**:

* **MUST** — the project. Every group does these.
* **SHOULD** — where your group goes deeper. Agree with your supervisor in the
  project plan which of these you commit to.
* **COULD** — optional and unscaffolded. If you take one on, you design it.

The last few sections are not analysis at all: dividing the work, risks, ethics,
and a list of questions nobody has answered yet. Read them in week one, not in
week nine.

**You are six or seven people.** The MUST sections are perhaps a week of work
for a pair. They are the starting point, not the project. Split into
sub-teams early — there is a suggested division near the end — and remember
that two people doing the same thing twice is the most common way a capstone
group wastes a month.

If you disagree with a choice made here, change it and say why when you
present. That is a better outcome than following it.

In [ ]:
# --- Setup --------------------------------------------------------------------
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone


def find_project_root(marker="src/capstone_tools.py"):
    """Locate the top of the cloned repository.

    Walks up from the current working directory, so the notebook works whether
    Jupyter was started in notebooks/, in the project root, or anywhere else
    inside the clone. Nothing here depends on where you put the folder.
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / marker).exists():
            return base
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the "
        "cloned repository, so that a 'src/capstone_tools.py' exists in one of "
        "the parent directories of " + str(Path.cwd())
    )


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import capstone_tools as ct

warnings.filterwarnings("ignore", category=FutureWarning)
ct.set_plot_style()

RANDOM_STATE = 0          # fixed so that your results are reproducible
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(exist_ok=True)

print("project root  :", ROOT)
print("data directory:", ct.DATA_DIR)
print("parameters:", ", ".join(ct.PARAMETER_NAMES))

## 1. The nine turbulence-model parameters

In [ ]:
info = ct.load_parameter_info()
info

## 2. The data: eighty outputs, not one

We work with **turbulent channel flow at $Re_\tau = 640$** — flow between two
parallel walls.

Each of the 1408 simulations produced two physical quantities of interest, each
sampled at **40 positions across the channel**. Two variables times forty
positions is the **80 CFD outputs** you can explain.

| symbol | name in the files | meaning |
|---|---|---|
| $p'_{\mathrm{rms}}$ | `p_rms` | pressure-fluctuation intensity — drives vibration and fatigue |
| $k$ | `k` | turbulent kinetic energy — the central quantity of the turbulence model |

The position is measured in **wall units** $y^+$: $y^+ \approx 1$ is right at the
wall, $y^+ \approx 600$ is the middle of the channel. The wall region is where
the interesting physics — and, as you will see, the interesting sensitivity —
lives.

We also provide six **scalar outputs** derived from those profiles (peak values,
wall values, integrals). Start with one of those, then come back to the profiles.

In [ ]:
CASE, RE_TAU = "channel", 640

data = ct.load_scalar_outputs(CASE, RE_TAU)
coords = ct.load_coordinates(CASE, RE_TAU)
point_cols = [f"point_{i:02d}" for i in range(40)]

print(f"{data.shape[0]} simulations x {len(ct.PARAMETER_NAMES)} parameters")
print("profile variables:", ", ".join(ct.PROFILE_VARIABLES),
      f"  x {len(point_cols)} positions = "
      f"{len(ct.PROFILE_VARIABLES) * len(point_cols)} outputs")
print("scalar outputs:", ", ".join(ct.SCALAR_OUTPUTS))
data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
for ax, var in zip(axes, ct.PROFILE_VARIABLES):
    prof = ct.load_profiles(CASE, var, RE_TAU)
    values = prof[point_cols].to_numpy()
    y = coords["y_plus"].to_numpy()
    for row in values[::25]:
        ax.plot(row, y, color="0.75", lw=0.6, alpha=0.7)
    ax.plot(values.mean(axis=0), y, color="crimson", lw=2.2)
    ax.set_xlabel(ct.VARIABLE_LABELS[var])
    ax.set_yscale("log")
    ax.set_title(var)
axes[0].set_ylabel(r"$y^+$ (distance from the wall)")
fig.suptitle(f"The four profile variables, {ct.case_label(CASE, RE_TAU)} "
             "(grey: individual simulations)")
fig.tight_layout()
fig.savefig(FIGDIR / "G2_01_profiles.png")
plt.show()

## 3. The benchmark

The Sobol index $S_i$ is the fraction of output variance explained by parameter
$i$ alone. Crucially for Group 2, the benchmark is provided **separately for
every one of the 80 outputs** — so you can check your explanation not just
overall, but position by position.

In [ ]:
bench_scalar = ct.load_sobol_scalar(CASE, RE_TAU)
bench_prof = ct.load_sobol_profiles(CASE, RE_TAU)

QOI = "p_rms_peak"

# Permutation importance and SHAP both include whatever a parameter does in
# combination with the others, so the TOTAL-order index is the like-for-like
# comparison. ST is the primary reference; S1 returns in section 13.
BENCHMARK = "ST"

bench_q = (bench_scalar[bench_scalar.output == QOI]
           .set_index("parameter").reindex(ct.PARAMETER_NAMES))

print("benchmark table (profiles):", bench_prof.shape)
bench_prof.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = bench_q[BENCHMARK].sort_values(ascending=False).index
idx = np.arange(len(order))
ax.bar(idx - 0.2, bench_q.loc[order, "ST"], width=0.4,
       yerr=bench_q.loc[order, "ST_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], label="$S_T$ (total order)")
ax.bar(idx + 0.2, bench_q.loc[order, "S1"], width=0.4,
       yerr=bench_q.loc[order, "S1_conf"], capsize=3,
       color=[ct.COLOURS[p] for p in order], alpha=0.45, label="$S_1$ (first order)")
ax.set_xticks(idx)
ax.set_xticklabels([ct.PRETTY[p] for p in order])
ax.set_ylabel("Sobol index")
ax.set_title(f"Benchmark from all 1408 CFD simulations — {QOI}")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "G2_02_benchmark.png")
plt.show()
print(f"sum of S1 = {bench_q.S1.sum():.3f}   sum of ST = {bench_q.ST.sum():.3f}")

---
## 4. MUST — a surrogate model and an honest error estimate

### 4a. Splitting the data correctly

The 1408 simulations are **not** 1408 independent random samples. They come from
a *Saltelli design*, built in 128 blocks of 11, and within a block the samples
differ in **only one of the nine parameters**.

So if you split rows at random, nearly every test simulation has a near-twin in
the training set and your test score is flattered. The dataset carries a
`design_group` column; keep each block on one side of the split.

In [ ]:
# ==========================================================================
# TODO - compare a random split with a grouped split
#
# Find out for yourselves how much a naive random split flatters the model.
#
# Fit the same model twice on the same data: once with an ordinary random split,
# once keeping each design_group together. Report the test score from each, and
# make the comparison fair - same model, same test fraction, same seed.
#
# Keep the two numbers; they show you understood the data before you modelled it.
#
# Hints:
#   * the dataset's structure is explained in DATA_DICTIONARY.md section 3
#   * RESOURCES.md section 2 lists the splitters that respect groups
#
# Later cells expect:
#   X, y, groups : the feature matrix, the target, and the design-group labels
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - fit the surrogate, measure it, and draw the parity plot
#
# Make the grouped split you will use throughout, fit a surrogate, and report
# RMSE, normalised RMSE and R2 on the test set. Then draw the parity plot.
#
# Choose the model settings yourselves and be ready to justify them. You will fit
# this model many times over in section 10, so think about cost as well as
# accuracy while you choose.
#
# Hints:
#   * RESOURCES.md sections 1 and 3
#   * normalised RMSE is RMSE divided by the spread of the test target
#
# Later cells expect:
#   X_train, X_test, y_train, y_test : the grouped split
#   train_idx, test_idx              : the corresponding index arrays
#   forest                           : the fitted surrogate
# ==========================================================================


# your code here

---
## 5. MUST — permutation feature importance

Shuffle one parameter column in the *test* set, so the parameter keeps its
distribution but loses its relationship with the output, and see how much the
score drops. A large drop means the model relied on that parameter.

In [ ]:
# ==========================================================================
# TODO - explain the surrogate, and compare with the benchmark
#
# Apply permutation feature importance to your fitted surrogate, then put your
# ranking next to the benchmark.
#
# Decide which data to measure the importance on, and how many repeats you need.
# Before plotting, work out whether the two quantities can share an axis; if they
# cannot, make that visible in the figure rather than quietly rescaling one.
#
# Hints:
#   * RESOURCES.md section 4
#   * keep a fixed parameter order and ct.COLOURS so all your figures match
#
# Later cells expect:
#   importance : pd.Series of permutation importances, indexed by parameter
#   spread     : pd.Series of the matching standard deviations
#   merged     : DataFrame indexed by parameter with columns 'permutation',
#                'S1', 'ST' and 'benchmark' (= merged[BENCHMARK])
# ==========================================================================


# your code here

## 6. SHOULD — put a number on the agreement

Permutation importance is a *drop in $R^2$*; a Sobol index is a *fraction of
variance*. They are not on the same scale, so compare **rankings**, not values.

In [ ]:
# ==========================================================================
# TODO - turn the comparison into a number
#
# Write a function comparing an importance vector against a benchmark vector,
# returning at least a rank correlation and the top-k overlap for a few k.
#
# Apply it, then compare the numbers with your impression from the bar charts. If
# a measure disagrees with your eyes, decide which is misleading you. Look at the
# benchmark values of the five least important parameters before answering, and be
# ready to defend whichever measure you quote as the right one for the question.
#
# Hints:
#   * RESOURCES.md section 5
#
# Later cells expect:
#   rank_agreement(scores, reference, ks=(2, 3, 4)) -> dict
#       reused in later sections, so keep this signature
# ==========================================================================


# your code here

## 7. SHOULD — error bars, and which parameters are genuinely separated

Two different sources of spread: the random shuffling inside
`permutation_importance`, and the choice of which simulations ended up in the
training set. The second is usually larger and is the one that tells you whether
the conclusion would survive a different set of simulations.

In [ ]:
# ==========================================================================
# TODO - put error bars on the importance scores
#
# Repeat the fit-and-explain procedure over several different grouped splits and
# report the spread. There are two distinct sources of variability here; identify
# both, decide which your error bars should show, and say why.
#
# State which parameters are separated by more than that spread.
#
# Hints:
#   * sklearn.base.clone for a fresh model each repeat
#   * collect each repeat as a pd.Series indexed by parameter
#
# Later cells expect:
#   summary : DataFrame indexed by parameter with columns ['mean', 'std']
# ==========================================================================


# your code here

---
## 8. SHOULD — a second explanation: SHAP

Permutation importance asks *"how much worse does the model get without this
parameter?"*. **SHAP** asks a different question: *"for this particular
simulation, how much did each parameter push the prediction away from the
average?"*

SHAP therefore gives one value per parameter **per simulation**, not a single
global number. Averaging the absolute values over all simulations gives a global
importance that can be ranked — but the per-simulation detail is what makes SHAP
worth the extra effort.

For tree models there is an exact, fast algorithm (`TreeExplainer`), so this is
cheap here.

In [ ]:
try:
    import shap
    HAVE_SHAP = True
except ImportError:
    HAVE_SHAP = False
    print("shap is not installed - run:  pip install shap")

# ==========================================================================
# TODO - a second explanation: SHAP
#
# Compute SHAP values for your test set. The result is one value per simulation
# per parameter - a matrix, not a vector - so you will need to reduce it to a
# single global importance per parameter.
#
# How you reduce it is a real decision: try the obvious summary and the one that
# accounts for sign, see what each gives, and keep the one you can justify.
#
# Guard the cell so the notebook still runs for someone without shap installed.
#
# Hints:
#   * RESOURCES.md section 4 - there is a fast exact method for tree models
#
# Later cells expect:
#   shap_values    : array of shape (n_test, 9)
#   merged['shap'] : your global SHAP importance per parameter
# ==========================================================================


# your code here

In [ ]:
# ==========================================================================
# TODO - compare the three answers
#
# Plot permutation importance, SHAP and the benchmark side by side, each on its
# own axis, and print the rankings and agreement numbers.
#
# Then answer the section's question: do the two explainability methods agree with
# each other, and with the benchmark? Name any parameter they disagree about.
#
# One more thing to settle before you present this: if they do agree, is that
# evidence that both are right, or could they share an assumption that makes them
# agree while both being wrong? Section 13 is relevant.
#
# Hints:
#   * reuse rank_agreement() from section 6
# ==========================================================================


# your code here

### What SHAP shows that a bar chart cannot

The beeswarm plot below has one dot per simulation per parameter. The horizontal
position is that parameter's contribution to that prediction; the colour is the
parameter's *value*. A clean colour gradient from left to right means the effect
is monotonic — high values of the parameter consistently push the output one way.

In [ ]:
# ==========================================================================
# TODO - the SHAP summary plot
#
# Draw the beeswarm summary plot, then read it rather than just including it.
#
# For each of your important parameters: is the effect monotonic, and in which
# direction? Does any parameter push the prediction in different directions for
# different simulations, and what would that mean physically?
#
# Hints:
#   * RESOURCES.md section 4; pass show=False so you can save it yourself
# ==========================================================================


# your code here

## 9. SHOULD — a trap: importance measured on training data

`permutation_importance` can be computed on either set. On the **training** set
it tells you what the model *used to fit*; on the **test** set it tells you what
the model actually needs *to generalise*.

A flexible model can latch onto a parameter that carries no real signal, and the
training-set version will happily report that as importance. This is one of the
most common mistakes in applied explainable AI, and it is easy to demonstrate.

In [ ]:
# ==========================================================================
# TODO - a trap: importance measured on the training data
#
# Permutation importance can be computed on either the training or the test data,
# and the two answer different questions.
#
# Compute both, show them side by side, and explain the difference. Look
# specifically at the parameters the benchmark says are negligible, and separately
# at the dominant ones - they do not behave the same way, and working out why is
# the point of the section.
#
# Then state which one should be reported, and what could go wrong if someone
# reported the other.
#
# Hints:
#   * the only change is which data you pass in
# ==========================================================================


# your code here

---
## 10. SHOULD — the ranking is not one ranking: explaining many outputs

Everything so far explained **one** number. But the CFD study produced 80, and
there is no reason the same parameter should control all of them.

We now fit one surrogate per output and explain each one. This is the step where
Group 2's work becomes genuinely different from a single-output study.

The requirement asks for **several** of the 80 outputs, so start with a handful
of positions and check how long one takes. Running the complete set of 80 is an
optional extension - worth doing, but only once the pipeline works.

In [ ]:
# ==========================================================================
# TODO - explain many of the 80 outputs
#
# Everything so far explained one number. The study produced 80, and there is no
# reason the same parameter controls all of them.
#
# Write a function that, for a given set of outputs, fits a surrogate and computes
# importance for each, returning one tidy DataFrame. Record each surrogate's
# accuracy too - you will need it to know where the explanation can be trusted.
#
# Three decisions worth making deliberately:
#
#   * whether to use the same split for every output, and what breaks if you do
#     not;
#   * whether to normalise each output's importances, what that buys you, and what
#     it hides;
#   * how to keep the cost manageable - time one output before you loop.
#
# The requirement asks for several of the 80. Run the full set once the pipeline
# works, and cache the result to disk.
#
# Hints:
#   * ct.load_profiles(case, variable, re_tau) gives the 1408 x 40 table
#   * ct.PROFILE_VARIABLES lists the variables
#
# Later cells expect:
#   explain_many_outputs(...) -> DataFrame with columns
#          ['variable', 'point_index', 'parameter', 'importance', 'R2']
#   many : the result for the channel case
# ==========================================================================


# your code here

### How much does the ranking actually change between outputs?

In [ ]:
# ==========================================================================
# TODO - how often does the ranking actually change?
#
# Across the outputs you explained, count how many times each parameter comes out
# on top. One line of pandas, but it answers "is there one ranking, or many?"
# directly - and the answer decides how you should present everything that
# follows.
# ==========================================================================


# your code here

## 11. SHOULD — the summary figure an engineer would actually use

A heatmap of parameter importance against output. Rows are parameters, columns
are the four physical variables; the colour is importance averaged along the
profile.

In [ ]:
# ==========================================================================
# TODO - the summary figure an engineer would use
#
# Build one figure that answers "which parameters should I worry about?" at a
# glance, across outputs.
#
# A heatmap is the obvious choice but not the only one. Whatever you choose, make
# it legible from the back of a room, and make sure the colour scale cannot
# mislead - think about what a reader concludes from a saturated cell when the
# underlying surrogate fitted that output badly.
# ==========================================================================


# your code here

## 12. COULD — where in the flow does each parameter matter?

This section is optional, but it is the most rewarding thing in the Group 2
project and we recommend it strongly if you have the time.

It is the figure that answers the second half of Group 2's question. The
profile-averaged heatmap hid the spatial structure; here we keep it.

Left: what your AI workflow recovered. Right: the CFD benchmark. If the two
panels have the same shape, your workflow has reproduced a result that cost 1408
CFD simulations.

In [ ]:
# ==========================================================================
# TODO - where in the flow does each parameter matter?
#
# Optional, but the most rewarding figure in the Group 2 project.
#
# For each variable, plot importance against position across the flow, with your
# AI result in one panel and the CFD benchmark beside it. The interesting physics
# is near the wall, so choose your axis accordingly.
#
# Make sure the two panels are comparable - if you normalised your own
# importances, the benchmark needs the same treatment.
#
# Then compare them honestly. Where do they agree, where do they not, and does the
# disagreement coincide with anywhere your surrogate was weak?
#
# Hints:
#   * pivot your results to index='point_index', columns='parameter'
#   * bench_prof holds the benchmark at every point
#   * plot only the handful of parameters that matter, or it is unreadable
# ==========================================================================


# your code here

## 13. SHOULD — do the parameters interact?

$S_1$ is a parameter acting alone; $S_T$ includes every interaction it takes part
in. The difference matters because permutation importance and SHAP both *include*
interaction effects, so if interactions were strong you would expect them to
disagree with $S_1$.

In [ ]:
# ==========================================================================
# TODO - do the parameters interact?
#
# Compare the first-order and total-order indices and look at the column sums.
#
# Then answer: given what you find, should permutation importance and SHAP have
# been expected to agree with the first-order indices? And would your method
# comparison in section 8 have come out the same way on a problem where the sums
# were very different? That second question is the one that tells you how far your
# conclusions generalise.
#
# Hints:
#   * both columns are already in `merged`
# ==========================================================================


# your code here

---
# Optional extensions (COULD)

Sections 4 to 11 and 13 cover the MUST and SHOULD requirements. Section 12 was
already optional, and everything from here on is too. Pick whatever interests
your group; you are not expected to do all of it.

One Could in the project description has no section of its own below, because
it is about the repository rather than about an analysis: leaving behind a
**tidy, documented repository** that could be pointed at a new parameter-study
dataset with little modification. If you lifted the spine into `src/` as
suggested above, you are most of the way there already — and it costs almost
nothing if you decide on it early.

## 14. COULD — compressing the 80 outputs into a few modes with PCA

Modelling 80 outputs means 80 surrogates and 80 explanations, which is a lot of
result for an audience to digest. **Principal component analysis** asks whether
the 80 numbers really move independently, or whether the whole profile shifts in
a few characteristic patterns.

The two variables have wildly different magnitudes ($p'_{\mathrm{rms}} \approx
2.5$, $k \approx 10^{-4}$), so we standardise every column first — otherwise PCA
would describe only the largest-numbered variable.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

### Which parameters control each mode?

Now train a surrogate for each principal-component score and explain *that*.
Instead of 80 explanations you get three, each describing a whole pattern of
variation across the flow.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 15. COULD — not just how much, but in which direction

Importance tells you a parameter matters. A **partial dependence plot** tells you
*what it does*: how the predicted output changes as that parameter sweeps across
its range, averaging over all the others.

This is often the plot an engineer actually wants, because it says which way to
turn the dial.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 16. COULD — does the spatial pattern hold at other Reynolds numbers?

The dataset contains the same study at $Re_\tau$ = 180, 395, 640 and 1020. The
benchmark indices show that the *shape* of the sensitivity profile is
preserved, but that the balance between the two dominant parameters shifts with
Reynolds number. Can your workflow see that shift?

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 17. COULD — a different geometry

The annular-flow case is the same turbulence model in the curved gap between a
rod and an outer cylinder. Switching cases is a one-line change — which is the
point of writing the pipeline cleanly.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

## 18. COULD — an interactive dashboard

Everything you have produced is a static picture. A small interactive tool
would let an engineer choose a flow case, a variable and a position, and watch
the parameter importance update — far more useful than a folder of PNGs.

`ipywidgets` is the least-effort route inside a notebook; `streamlit` or `dash`
if you want something that runs as its own app. Keep it small: one or two
controls that genuinely help someone read your results.

In [ ]:
# ==========================================================================
# TODO - optional extension
#
# This section is deliberately unscaffolded. Decide whether your group
# wants it, agree the approach with your supervisor, and build it
# yourselves. The markdown above says what it should achieve.
# ==========================================================================


# your code here

---
# Running the project

The sections above are the technical work. These are the things that decide
whether a capstone group finishes well, and they are assessed too.

## Dividing the work

### What blocks what

Group 2 has a useful property: the expensive part does not depend on the rest.

    sections 4-6   the spine: split, surrogate, importance, agreement
      |            ONE pair, first week.
      |
      +-- section 7    error bars                    independent
      +-- section 8    SHAP                          independent
      +-- section 9    training vs test importance   independent
      +-- section 13   interactions                  independent, ~1 hour
      +-- section 15   partial dependence            independent

    section 10   the sweep over the 80 outputs
      |          needs NOTHING from the spine - it builds its own split.
      |          It is the slowest thing here, so start it on day one.
      |
      +-- section 11   heatmap                       needs 10
      +-- section 12   spatial map                   needs 10
      +-- sections 16, 17   Reynolds sweep, annular  needs 10

    section 14   PCA         needs nothing
    section 18   dashboard   needs whatever you have built

So one pair can start the long-running sweep immediately while another pair
builds the spine. Nobody has to wait for anybody in week one.

### Suggested split

* **Methods** - sections 7, 8, 9, 13, 15. Owns the comparison between
  permutation importance, SHAP and partial dependence. Answers "which
  explanation should an engineer trust?".
* **Across outputs** - sections 10, 11, 12. Owns the sweep, the heatmap and the
  spatial map. Answers "does the answer depend on what you ask about?". This
  pair also owns caching, because this is the expensive part.
* **Structure** - sections 14, 16, 17, 18. Owns the PCA route and the extensions
  to other Reynolds numbers and the annular case.

With seven people the spare person is best used on shared infrastructure -
caching, the results table, the figures everyone needs - not as a third member
of one pair.


Three rules that save groups a lot of pain:

1. **Lift the spine out of the notebook.** As soon as sections 4-6 work, move
   that code into a small module under `src/` — the split, the fitted model,
   the importance table — so every sub-team imports the same objects instead of
   re-running the notebook top to bottom and quietly getting a different split.
   This single step is what turns a sequential notebook into parallel work.
2. **One shared split, fixed early.** If different sub-teams use different
   train/test splits, none of your numbers can be compared with any other.
   Fix `RANDOM_STATE` and the split once, in writing.
3. **Everything in git, from week one.** Six people editing one notebook is the
   classic failure. Split your work into separate notebooks or modules that
   import from `src/`, and merge deliberately.

## Requirements, and renegotiating them

Your project plan lists the MoSCoW requirements you have agreed with the
supervisor. Keep a table in the repository showing, for each requirement, its
status and the evidence — the figure or the number that demonstrates it. You
will need exactly this table for the final presentation, and filling it in as
you go costs nothing while reconstructing it in week nine costs a day.

If a Must starts looking unreachable, say so early and renegotiate. That is
explicitly allowed, and it is assessed far better than a requirement quietly
dropped. Risk taken knowingly and reported honestly is a good outcome; a
surprise in week nine is not.

## What could go wrong

Write these down in the project plan with a mitigation for each. Some that are
real for this project:

* **The cell that runs for an hour.** Several steps here fit one model per
  output. Time a single output before you loop over all of them, and cache
  results to disk so you never recompute them.
* **Results that cannot be compared.** Different splits, different random
  seeds, different normalisations between sub-teams. Fix them centrally.
* **A finding that disappears on re-run.** If a conclusion depends on one random
  seed, it is not a finding. Repeat anything you intend to present.
* **The deadline shape.** You get three weeks of full-time work and nothing
  before it. Whatever you want to present must exist in a rough form by the end
  of week 2.8, so that week 2.9 is for checking and explaining, not discovering.

## Ethics and societal impact

You will be asked about this, and for this project it is not an afterthought.

The data comes from a study of **turbulence-induced vibration in nuclear fuel
assemblies**: coolant flowing past fuel rods produces unsteady pressure that
shakes them, and over years that causes fretting wear and fatigue. The
turbulence-model parameters you are ranking feed predictions used in safety
assessments.

Questions worth a slide:

* **What is the cost of being wrong in each direction?** Your workflow says
  certain parameters do not matter, so an engineer stops calibrating them. If
  that judgement is wrong, the predicted vibration is wrong. Is the error more
  likely to be optimistic or pessimistic, and which is worse here?
* **Who is accountable** when a cheap surrogate replaces an expensive analysis
  and the cheap one is wrong — the person who built the surrogate, or the person
  who trusted it? What would you have to report alongside a ranking for the
  second person to use it responsibly?
* **Does a confident-looking output deserve confidence?** You will produce
  normalised importances that always sum to one, so they look equally certain
  for every output — including outputs where the surrogate fits badly. What
  would you add to the figure to stop a reader over-reading it?
* **The positive side.** The CFD campaign behind this dataset cost on the order
  of 2 x 10^5 CPU hours. If a surrogate reaches the same conclusion from a
  fraction of the simulations, that is a real saving in energy and emissions as
  well as money. Estimate it — with your assumptions stated — rather than
  asserting it.
* **Automation and judgement.** If this workflow became routine, what expertise
  stops being exercised, and does that matter?

A good answer is specific to this project and admits a trade-off. A generic
paragraph about "AI bias" will not do.

## Open questions

Nobody has answered these on this dataset, including your supervisor. They are
offered because a capstone is more interesting when some of it is real. Each is
a reasonable size for a pair in the full-time weeks, and none has a known
answer, so a careful negative result is a perfectly good outcome.

1. **Is a bad explanation predictable from a bad surrogate?** You fit one model
   per output, and some fit better than others. Does explanation quality — the
   agreement with the benchmark — track surrogate quality point by point? If it
   does, you can warn a reader where not to trust the map, without ever seeing
   the benchmark.
2. **Do the methods agree where the model is good?** Permutation importance and
   SHAP mostly agree here. Do they disagree more where the surrogate is weak? A
   disagreement between two explanations would then be a usable warning sign.
3. **Is explaining the modes the same as averaging the explanations?** You can
   explain each of the 80 outputs and average, or compress with PCA and explain
   the modes. Do the two give the same ranking? If not, which is the more honest
   summary, and why?
4. **Physics or data quality?** The annular case ranks the parameters
   differently from the channel. Is that because the flow is genuinely
   different, or because the surrogate fits that case differently well? Design a
   check that separates the two explanations.
5. **What does a near-one-dimensional output space mean?** You may find that a
   single mode explains almost all the variation across all 80 outputs. Is that
   a property of the flow, or of the turbulence model being unable to vary its
   predictions independently? What would distinguish those two readings?


If you take one on, agree it with your supervisor first, and treat it the way
you would treat a Should: state the question, say how you will answer it, and
report what you actually found rather than what you hoped for.

---
# 19. What you actually hand in

This course has **no written report**. You are assessed on three things:

| when | deliverable |
|---|---|
| before the Christmas break | a **project plan**, agreed with the supervisor |
| week 9 | the **AI solution** — code, results, figures, saved models |
| week 10 | a **final presentation** to your supervisor and the assessors |

So the work below is not material for a report; it is material for a talk and a
clean repository. Plan accordingly: a figure that needs three paragraphs of
explanation is a bad slide.

### For the project plan

Decide early which SHOULD items your group will commit to, and say which ones
you will not do. You have six or seven people — split the work by section, not by
"everyone tries the same model". Natural division for Group 2: one pair on the
method comparison (permutation vs SHAP vs partial dependence), one pair on the
sweep across all 80 outputs and the spatial map, one pair on PCA, everyone on
interpretation.

### For the week-9 solution

* a repository that runs end to end from a clean clone;
* your notebooks with outputs, so results can be seen without re-running;
* the figures you will actually present, saved as files;
* a short README stating what you did, what you found, and what you did not do.

### For the final presentation

A suggested spine, roughly one slide each:

1. **The question.** Which explanation should an engineer trust, and does the
   answer depend on where in the flow you look?
2. **The data and the trap.** The Saltelli design, and why a random split
   flatters you. Show the two R² numbers from section 4a.
3. **The surrogate works.** Parity plot, with RMSE / nRMSE / R² and your split
   stated on the slide.
4. **Three methods, one question.** Permutation importance, SHAP and the Sobol
   benchmark side by side. Say clearly that they are measured in different units,
   so only the ranking is comparable.
5. **The trap that catches people.** Training-set versus test-set importance.
6. **The spatial map**, if you did section 12 — the strongest slide available
   to you: the AI panel next to the benchmark panel.
7. **Limits.** See below.
8. **Conclusion** in one sentence that answers slide 1.

### Be honest about the limits — it earns marks, it does not lose them

Specific things that are true of this dataset:

* seven of the nine parameters have near-zero importance, so their ranking is
  noise;
* normalising importance to sum to one makes a weak output look confident;
* the surrogate is only as good as its R² at that point in the flow, and that is
  worst near the wall;
* the explainability methods largely agree here *because* the response is nearly
  additive — that would not be guaranteed for a problem with strong interactions;
* one geometry and one Reynolds number, unless you did sections 16 and 17.